# Your first LiDAR map with `pip install lidarslam`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/lidar_slam/blob/develop/examples/colab/first_map.ipynb)

This notebook builds a verified point-cloud map from a public Livox MID-360 rosbag2 with
[lidar_slam](https://github.com/rsasaki0109/lidar_slam), without ROS 2: the `lidarslam`
wheel brings RKO-LIO odometry, the pose-graph backend and the Autoware map export.

It takes about 5 minutes on a standard Colab CPU runtime: 1 minute to install, 1 to 2 minutes
to download the 517 MB recording, 2 minutes to map it.

Data: Livox MID-360 public driving recording `rosbag2_2024_04_16-14_17_01`
([Zenodo 14841855](https://zenodo.org/records/14841855), CC BY 4.0).

## 1. Install

In [ ]:
!pip install -q lidarslam pypcd4
!lidarslam-map --version

## 2. Download the recording

In [ ]:
import pathlib, urllib.request, zipfile

URL = "https://zenodo.org/records/14841855/files/rosbag2_2024_04_16-14_17_01.zip?download=1"
data = pathlib.Path("data")
data.mkdir(exist_ok=True)
archive = data / "rosbag2_2024_04_16-14_17_01.zip"
if not archive.exists():
    def progress(blocks, block_size, total):
        print(f"\r{min(blocks * block_size, total) / 1e6:.0f} / {total / 1e6:.0f} MB", end="")
    urllib.request.urlretrieve(URL, archive, progress)
    print()
assert archive.stat().st_size == 517_088_133, "unexpected archive size"
with zipfile.ZipFile(archive) as zf:
    zf.extractall(data)
bag = next(p.parent for p in data.rglob("metadata.yaml"))
print("rosbag2:", bag)

## 3. Map it

`lidarslam-map start` detects the sensors, picks the MID-360 profile, runs the odometry and the
backend, writes the Autoware map bundle and verifies it. `--engine core` runs everything from the
wheel; no ROS 2 is installed here.

In [ ]:
!lidarslam-map start {bag} --engine core --yes --viewer none --output-dir run

## 4. Look at the map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pypcd4 import PointCloud

points = PointCloud.from_path("run/map/map.pcd").numpy(("x", "y", "z"))
trajectory = np.loadtxt("run/map/traj_corrected.tum")
print(f"{len(points):,} map points, {len(trajectory)} poses")

sample = points[np.random.default_rng(0).choice(len(points), min(len(points), 400_000), replace=False)]
low, high = np.percentile(sample[:, 2], [2, 98])
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(sample[:, 0], sample[:, 1], c=np.clip(sample[:, 2], low, high), s=0.05, cmap="turbo")
ax.plot(trajectory[:, 1], trajectory[:, 2], color="black", linewidth=1.5, label="trajectory")
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Map, top view (colour: height)")
ax.legend()
plt.show()

In [ ]:
# Interactive 3-D view (drag to orbit). Thinned to keep the notebook light.
import plotly.graph_objects as go

view = sample[::4]
go.Figure([
    go.Scatter3d(x=view[:, 0], y=view[:, 1], z=view[:, 2], mode="markers",
                 marker=dict(size=1, color=np.clip(view[:, 2], low, high), colorscale="Turbo"),
                 name="map"),
    go.Scatter3d(x=trajectory[:, 1], y=trajectory[:, 2], z=trajectory[:, 3], mode="lines",
                 line=dict(color="black", width=4), name="trajectory"),
]).update_layout(scene=dict(aspectmode="data"), height=700, margin=dict(l=0, r=0, t=0, b=0)).show()

## 5. Take the map with you

`run/map` holds the Autoware-compatible bundle: `pointcloud_map/` tiles,
`map_projector_info.yaml` and `lanelet2_map.osm`, plus `map.pcd`, the trajectories and the
verification receipt (`first_map_validation_receipt.md`).

In [ ]:
!cat run/map/first_map_validation_receipt.md | head -20
!cd run && zip -qr map_bundle.zip map -x "map/*.log"
try:
    from google.colab import files
    files.download("run/map_bundle.zip")
except ImportError:
    print("run/map_bundle.zip")

## Your own bag

Upload a rosbag2 directory (the folder with `metadata.yaml`, zipped) with the file browser on the
left, unzip it, and run `lidarslam-map doctor <bag>` first: it names the LiDAR and IMU topics it
found and the profile it would use. Then `lidarslam-map start <bag> --engine core --yes
--viewer none`. See [Getting Started](https://rsasaki0109.github.io/lidar_slam/getting-started.html)
for sensors, profiles and extrinsics.